In [1]:
import sys

print(sys.executable)
print(sys.version)

c:\Users\s.peikmortazavi\credit-risk-intelligence-copilot\.venv\Scripts\python.exe
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [3]:
PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "application_train.csv"
)

print(DATA_PATH)
print("File exists:", DATA_PATH.exists())

c:\Users\s.peikmortazavi\credit-risk-intelligence-copilot\data\raw\application_train.csv
File exists: True


Load Dataset

In [4]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
df.columns.tolist()

['SK_ID_CURR',
 'TARGET',
 'NAME_CONTRACT_TYPE',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'CNT_CHILDREN',
 'AMT_INCOME_TOTAL',
 'AMT_CREDIT',
 'AMT_ANNUITY',
 'AMT_GOODS_PRICE',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'REGION_POPULATION_RELATIVE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'DAYS_REGISTRATION',
 'DAYS_ID_PUBLISH',
 'OWN_CAR_AGE',
 'FLAG_MOBIL',
 'FLAG_EMP_PHONE',
 'FLAG_WORK_PHONE',
 'FLAG_CONT_MOBILE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'OCCUPATION_TYPE',
 'CNT_FAM_MEMBERS',
 'REGION_RATING_CLIENT',
 'REGION_RATING_CLIENT_W_CITY',
 'WEEKDAY_APPR_PROCESS_START',
 'HOUR_APPR_PROCESS_START',
 'REG_REGION_NOT_LIVE_REGION',
 'REG_REGION_NOT_WORK_REGION',
 'LIVE_REGION_NOT_WORK_REGION',
 'REG_CITY_NOT_LIVE_CITY',
 'REG_CITY_NOT_WORK_CITY',
 'LIVE_CITY_NOT_WORK_CITY',
 'ORGANIZATION_TYPE',
 'EXT_SOURCE_1',
 'EXT_SOURCE_2',
 'EXT_SOURCE_3',
 'APARTMENTS_AVG',
 'BASEMENTAREA_AVG',
 'YEARS_BEGINEXPLUATATION_A

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 122 entries, SK_ID_CURR to AMT_REQ_CREDIT_BUREAU_YEAR
dtypes: float64(65), int64(41), str(16)
memory usage: 286.2 MB


In [8]:
print(df["TARGET"].value_counts())
print(df["TARGET"].value_counts(normalize=True))

TARGET
0    282686
1     24825
Name: count, dtype: int64
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


In [9]:
default_rate = df["TARGET"].mean()

print(f"Default rate: {default_rate:.2%}")

Default rate: 8.07%


In [10]:
ID_COL = "SK_ID_CURR"
TARGET = "TARGET"

print("Rows:", len(df))
print("Unique customers:", df[ID_COL].nunique())
print("Duplicate IDs:", df[ID_COL].duplicated().sum())
print("Missing target:", df[TARGET].isna().sum())

Rows: 307511
Unique customers: 307511
Duplicate IDs: 0
Missing target: 0


In [11]:
categorical_cols = df.drop(
    columns=[ID_COL, TARGET]
).select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_cols = df.drop(
    columns=[ID_COL, TARGET]
).select_dtypes(
    exclude=["object", "category"]
).columns.tolist()

print("Numerical features:", len(numerical_cols))
print("Categorical features:", len(categorical_cols))

Numerical features: 104
Categorical features: 16


C:\Users\s.peikmortazavi\AppData\Local\Temp\ipykernel_21784\263192491.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  ).select_dtypes(


In [12]:
categorical_cols

['NAME_CONTRACT_TYPE',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'OCCUPATION_TYPE',
 'WEEKDAY_APPR_PROCESS_START',
 'ORGANIZATION_TYPE',
 'FONDKAPREMONT_MODE',
 'HOUSETYPE_MODE',
 'WALLSMATERIAL_MODE',
 'EMERGENCYSTATE_MODE']

In [13]:
df[categorical_cols].nunique().sort_values(
    ascending=False
)

ORGANIZATION_TYPE             58
OCCUPATION_TYPE               18
NAME_INCOME_TYPE               8
NAME_TYPE_SUITE                7
WALLSMATERIAL_MODE             7
WEEKDAY_APPR_PROCESS_START     7
NAME_FAMILY_STATUS             6
NAME_HOUSING_TYPE              6
NAME_EDUCATION_TYPE            5
FONDKAPREMONT_MODE             4
HOUSETYPE_MODE                 3
CODE_GENDER                    3
FLAG_OWN_CAR                   2
NAME_CONTRACT_TYPE             2
FLAG_OWN_REALTY                2
EMERGENCYSTATE_MODE            2
dtype: int64

In [14]:
missing_summary = (
    df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .to_frame("missing_pct")
)

missing_summary.head(20)

,missing_pct
COMMONAREA_AVG,69.872297
COMMONAREA_MODE,69.872297
COMMONAREA_MEDI,69.872297
NONLIVINGAPARTMENTS_MEDI,69.432963
NONLIVINGAPARTMENTS_MODE,69.432963
NONLIVINGAPARTMENTS_AVG,69.432963
FONDKAPREMONT_MODE,68.386172
LIVINGAPARTMENTS_AVG,68.354953
LIVINGAPARTMENTS_MEDI,68.354953
LIVINGAPARTMENTS_MODE,68.354953


In [15]:
print(
    "Features with missing values:",
    (missing_summary["missing_pct"] > 0).sum()
)

print(
    "Features with >50% missing:",
    (missing_summary["missing_pct"] > 50).sum()
)

Features with missing values: 67
Features with >50% missing: 41


In [16]:
nunique = df.drop(
    columns=[ID_COL, TARGET]
).nunique()

constant_features = nunique[
    nunique <= 1
].index.tolist()

print("Constant features:", constant_features)

Constant features: []


In [17]:
print("=" * 40)
print("HOME CREDIT DATA SUMMARY")
print("=" * 40)

print(f"Observations:        {len(df):,}")
print(f"Features:            {df.shape[1] - 2}")
print(f"Numerical features:  {len(numerical_cols)}")
print(f"Categorical features:{len(categorical_cols)}")
print(f"Default rate:        {df[TARGET].mean():.2%}")
print(f"Duplicate IDs:       {df[ID_COL].duplicated().sum()}")
print(f"Missing targets:     {df[TARGET].isna().sum()}")
print(
    f">50% missing:        "
    f"{(missing_summary['missing_pct'] > 50).sum()}"
)

HOME CREDIT DATA SUMMARY
Observations:        307,511
Features:            120
Numerical features:  104
Categorical features:16
Default rate:        8.07%
Duplicate IDs:       0
Missing targets:     0
>50% missing:        41


In [18]:
X = df.drop(columns=[ID_COL, TARGET])
y = df[TARGET]

customer_ids = df[ID_COL]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target rate:", y.mean())

X shape: (307511, 120)
y shape: (307511,)
Target rate: 0.08072881945686496


307,511
   │
   ├──── Train       70% ≈ 215k
   │
   ├──── Validation  15% ≈ 46k
   │
   └──── Test        15% ≈ 46k

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42,
)


X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42,
)

In [20]:
splits = {
    "Train": y_train,
    "Validation": y_valid,
    "Test": y_test,
}

for name, target in splits.items():
    print(
        f"{name:<12} "
        f"N={len(target):>7,} | "
        f"Default Rate={target.mean():.2%}"
    )

Train        N=215,257 | Default Rate=8.07%
Validation   N= 46,127 | Default Rate=8.07%
Test         N= 46,127 | Default Rate=8.07%


In [21]:
categorical_features = (
    X_train
    .select_dtypes(include=["object", "category"])
    .columns
    .tolist()
)

numerical_features = (
    X_train
    .select_dtypes(exclude=["object", "category"])
    .columns
    .tolist()
)

print("Numerical:", len(numerical_features))
print("Categorical:", len(categorical_features))

Numerical: 104
Categorical: 16


C:\Users\s.peikmortazavi\AppData\Local\Temp\ipykernel_21784\4034691634.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  .select_dtypes(include=["object", "category"])


In [22]:
from credit_risk_copilot.ml.preprocessing import build_preprocessor

preprocessor = build_preprocessor(
    numerical_features=numerical_features,
    categorical_features=categorical_features,
)

X_train_processed = preprocessor.fit_transform(X_train)

X_valid_processed = preprocessor.transform(X_valid)
X_test_processed = preprocessor.transform(X_test)

In [23]:
print("Before preprocessing")
print("--------------------")
print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Test :", X_test.shape)

print("\nAfter preprocessing")
print("--------------------")
print("Train:", X_train_processed.shape)
print("Valid:", X_valid_processed.shape)
print("Test :", X_test_processed.shape)

Before preprocessing
--------------------
Train: (215257, 120)
Valid: (46127, 120)
Test : (46127, 120)

After preprocessing
--------------------
Train: (215257, 244)
Valid: (46127, 244)
Test : (46127, 244)


In [24]:
print(
    "NaNs after preprocessing:",
    np.isnan(X_train_processed.data).sum()
    if hasattr(X_train_processed, "data")
    else np.isnan(X_train_processed).sum()
)

NaNs after preprocessing: 0
